# XAI-SHAP Framework Tutorial

## Quick Start Guide for Explainable AI with SHAP

This notebook demonstrates the core functionality of the XAI-SHAP Visual Analytics Framework.

### Contents
1. Setup and Installation
2. Loading Data
3. Training Models
4. Generating SHAP Explanations
5. Visualizing Results
6. Fairness Analysis

## 1. Setup and Installation

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add project to path
import sys
sys.path.insert(0, '..')

# Import framework
from src.core.framework import XAIFramework
from src.visualization.plots import XAIVisualizer

print("✅ Libraries imported successfully!")

## 2. Loading Data

In [ ]:
# Create sample data
from sklearn.datasets import load_breast_cancer

# Load breast cancer dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['target'].value_counts())

df.head()

In [ ]:
# Initialize framework
framework = XAIFramework()

# Load data into framework
framework.load_data(
    data_path=df,
    target_column='target',
    test_size=0.2
)

print(f"Training samples: {len(framework.X_train)}")
print(f"Test samples: {len(framework.X_test)}")
print(f"Features: {len(framework.feature_names)}")

## 3. Training Models

In [ ]:
# Train XGBoost model
framework.train_model(
    model_type='xgboost',
    model_params={
        'n_estimators': 100,
        'max_depth': 5,
        'learning_rate': 0.1,
        'random_state': 42
    }
)

print(f"Model trained: {type(framework.model).__name__}")

In [ ]:
# Evaluate model
from sklearn.metrics import classification_report, accuracy_score

y_pred = framework.model.predict(framework.X_test)
accuracy = accuracy_score(framework.y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(framework.y_test, y_pred))

## 4. Generating SHAP Explanations

In [ ]:
# Generate SHAP values
framework.explain_model(n_samples=100)

print(f"SHAP values shape: {framework.shap_values.shape}")

In [ ]:
# Calculate feature importance from SHAP values
mean_abs_shap = np.abs(framework.shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

print("Top 10 Important Features (by SHAP):")
print("-" * 40)
for i in range(10):
    idx = sorted_idx[i]
    print(f"{i+1}. {framework.feature_names[idx]}: {mean_abs_shap[idx]:.4f}")

## 5. Visualizing Results

In [ ]:
# Create feature importance bar chart
top_n = 15
top_indices = sorted_idx[:top_n]
top_features = [framework.feature_names[i] for i in top_indices]
top_importance = mean_abs_shap[top_indices]

plt.figure(figsize=(12, 8))
plt.barh(range(len(top_features)), top_importance[::-1], color='steelblue')
plt.yticks(range(len(top_features)), top_features[::-1])
plt.xlabel('Mean |SHAP Value|')
plt.title('Top 15 Feature Importance (SHAP)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Summary Plot (simplified version)
import shap

# Use SHAP's built-in plotting
shap.summary_plot(
    framework.shap_values, 
    framework.X_test[:100],
    feature_names=framework.feature_names,
    show=True
)

In [ ]:
# Waterfall plot for single prediction
sample_idx = 0

# Get explanation for single sample
sample_shap = framework.shap_values[sample_idx]
sample_features = framework.X_test[sample_idx]

# Create waterfall data
sorted_contributions = sorted(
    zip(framework.feature_names, sample_shap, sample_features),
    key=lambda x: abs(x[1]),
    reverse=True
)[:10]

plt.figure(figsize=(10, 6))
names = [f"{x[0]}={x[2]:.2f}" for x in sorted_contributions]
values = [x[1] for x in sorted_contributions]
colors = ['red' if v < 0 else 'green' for v in values]

plt.barh(range(len(names)), values[::-1], color=colors[::-1])
plt.yticks(range(len(names)), names[::-1])
plt.xlabel('SHAP Value')
plt.title(f'Feature Contributions for Sample {sample_idx}')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Interactive Visualizations with Plotly

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Create interactive feature importance chart
importance_df = pd.DataFrame({
    'feature': framework.feature_names,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=True).tail(15)

fig = px.bar(
    importance_df, 
    x='importance', 
    y='feature',
    orientation='h',
    title='Feature Importance (Interactive)',
    color='importance',
    color_continuous_scale='Blues'
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Dependence plot for top feature
top_feature_idx = sorted_idx[0]
top_feature_name = framework.feature_names[top_feature_idx]

fig = px.scatter(
    x=framework.X_test[:100, top_feature_idx],
    y=framework.shap_values[:, top_feature_idx],
    color=framework.y_test[:100],
    title=f'SHAP Dependence Plot: {top_feature_name}',
    labels={'x': top_feature_name, 'y': 'SHAP Value', 'color': 'Target'}
)
fig.show()

## 7. Model Comparison

In [ ]:
# Compare multiple models
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

results = []

# XGBoost (already trained)
xgb_pred = framework.model.predict(framework.X_test)
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(framework.y_test, xgb_pred),
    'F1 Score': f1_score(framework.y_test, xgb_pred)
})

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(framework.X_train, framework.y_train)
rf_pred = rf.predict(framework.X_test)
results.append({
    'Model': 'Random Forest',
    'Accuracy': accuracy_score(framework.y_test, rf_pred),
    'F1 Score': f1_score(framework.y_test, rf_pred)
})

comparison_df = pd.DataFrame(results)
print("Model Comparison:")
print(comparison_df.to_string(index=False))

## 8. Summary

This tutorial covered:
- ✅ Loading and preprocessing data
- ✅ Training XGBoost model
- ✅ Generating SHAP explanations
- ✅ Visualizing feature importance
- ✅ Creating interactive plots
- ✅ Comparing models

### Next Steps
- Explore the interactive dashboard: `streamlit run src/dashboard/app.py`
- Try fairness analysis: `python examples/fairness_example.py`
- Generate reports: See `examples/basic_usage.py`